# UCDP/PRIO Armed Conflict Dataset (v26.1) - Country Code Cleaning

### Objective
This notebook provides a clean, explainable pipeline to convert **Gleditsch-Ward (GW) numeric state codes** into standardized **ISO3 alpha-3 codes** across all country-related columns in the UCDP/PRIO ACD dataset (`data/2026_ucdp-prio-acd-261.xlsx`).

### Key Challenges Solved
1. **Non-State Actors (Side B in Intrastate Conflicts):** In civil wars (Type 3 and 4), Side B is an insurgent or rebel group (e.g., FARC, ELN, IS), which has no GW code (`NaN`). The conversion explicitly keeps these as `NaN` without corrupting the column.
2. **Coalitions & Secondary Parties:** Columns such as `gwno_loc`, `gwno_a_2nd`, and `gwno_b_2nd` often contain comma-separated lists of codes (e.g., `"2, 200, 678"`). Each code is parsed, converted, and recombined.
3. **Historical Entities:** Historical GW codes that `country_converter` misses by default (such as Hyderabad `751`, North Yemen `678`, North Vietnam `816`, and Yugoslavia `345`) are mapped explicitly via a transparent lookup table.

In [4]:
import numpy as np
import pandas as pd
import country_converter as coco
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)

## 1. Load UCDP/PRIO ACD Dataset

In [7]:
# Robust project root resolution across environments (local Jupyter, IDE, MCP)
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path(r"C:\Users\stynz\Desktop\NOVSLO~2\fundamentals_of_data_science"),
    Path.home() / "Desktop" / "NovĂˇ sloĹľka (2)" / "fundamentals_of_data_science"
]

project_root = None
for root in candidate_roots:
    if (root / "data" / "2026_ucdp-prio-acd-261.xlsx").exists():
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError("Could not locate project root containing data/2026_ucdp-prio-acd-261.xlsx")

file_path = project_root / "data" / "2026_ucdp-prio-acd-261.xlsx"
print(f"Project root: {project_root}")
print(f"Reading from: {file_path}")
df_raw = pd.read_excel(file_path)
print(f"Loaded dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head(3)

Project root: C:\Users\stynz\Desktop\NOVSLO~2\fundamentals_of_data_science
Reading from: C:\Users\stynz\Desktop\NOVSLO~2\fundamentals_of_data_science\data\2026_ucdp-prio-acd-261.xlsx


Loaded dataset: 2816 rows, 28 columns


,conflict_id,location,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,territory_name,year,intensity_level,cumulative_intensity,type_of_conflict,start_date,start_prec,start_date2,start_prec2,ep_end,ep_end_date,ep_end_prec,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,region,version
0,200,Bolivia,Government of Bolivia,23,NaN,Popular Revolutionary Movement,719,NaN,2,NaN,1946,2,1,3,1946-07-18,1,1946-07-21,2,1,1946-07-21,NaN,145,NaN,NaN,NaN,145,5,26.1
1,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,1949,1,1,3,1946-07-18,1,1949-09-15,3,1,1949-09-15,NaN,145,NaN,NaN,NaN,145,5,26.1
2,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,1952,1,1,3,1946-07-18,1,1952-04-09,1,1,1952-04-12,NaN,145,NaN,NaN,NaN,145,5,26.1


## 2. Define Country Converter & Historical Code Overrides

Gleditsch-Ward (GW) numbering includes historical states that may not exist in modern ISO-3166 lists. We define an explicit dictionary to cleanly map them.

In [8]:
# Initialize country_converter instance
cc = coco.CountryConverter()

# Explicit overrides for historical GW codes that country_converter cannot resolve
HISTORICAL_GW_TO_ISO3 = {
    316: "RUS",  # USSR / Russian SFSR
    345: "YUG",  # Yugoslavia (former)
    678: "YEM",  # Yemen Arab Republic (North Yemen)
    680: "YEM",  # People's Democratic Republic of Yemen (South Yemen)
    751: "IND",  # Hyderabad (annexed into India in 1948)
    816: "VNM",  # Democratic Republic of Vietnam (North Vietnam)
    817: "VNM",  # Republic of Vietnam (South Vietnam)
}

def convert_single_gw_code(code_val) -> str:
    """
    Converts a single GW numeric code (or string representation of one) into ISO3.
    Returns None if missing, invalid, or zero.
    """
    if pd.isna(code_val):
        return None
    
    # Clean string and convert to integer
    try:
        code_int = int(float(str(code_val).strip()))
    except (ValueError, TypeError):
        return None
        
    if code_int <= 0:
        return None
        
    # 1. Check manual historical dictionary
    if code_int in HISTORICAL_GW_TO_ISO3:
        return HISTORICAL_GW_TO_ISO3[code_int]
        
    # 2. Convert via country_converter
    res = cc.convert(names=code_int, src="GWcode", to="ISO3", not_found=None)
    return res if res and res != "not found" else None

def convert_gw_cell(val):
    """
    Handles single codes, floats, and comma-separated lists of codes.
    E.g.:
      145       -> "BOL"
      "28, 3"   -> "GBR, USA"
      np.nan    -> np.nan
    """
    if pd.isna(val):
        return np.nan
        
    val_str = str(val).strip()
    if not val_str or val_str in ['0', 'nan', 'None']:
        return np.nan
        
    # Case A: Comma-separated list of codes
    if "," in val_str:
        raw_codes = [c.strip() for c in val_str.split(",")]
        converted = [convert_single_gw_code(c) for c in raw_codes]
        valid_codes = [c for c in converted if c is not None]
        return ", ".join(valid_codes) if valid_codes else np.nan
        
    # Case B: Single code
    res = convert_single_gw_code(val_str)
    return res if res is not None else np.nan

## 3. Apply Conversion Across All GW Columns

We create explicit `iso3_*` columns corresponding to each `gwno_*` column.

In [9]:
df_clean = df_raw.copy()

gw_columns = [
    'gwno_a',      # Primary actor Side A (usually Government)
    'gwno_a_2nd',  # Supporting actors Side A
    'gwno_b',      # Primary actor Side B (State if interstate, NaN if rebel group)
    'gwno_b_2nd',  # Supporting actors Side B
    'gwno_loc'     # Conflict location
]

for col in gw_columns:
    if col in df_clean.columns:
        target_col = col.replace('gwno_', 'iso3_')
        df_clean[target_col] = df_clean[col].apply(convert_gw_cell)
        print(f"Converted {col} -> {target_col} (Non-null: {df_clean[target_col].notna().sum()} / {len(df_clean)})")

Converted gwno_a -> iso3_a (Non-null: 2816 / 2816)


Converted gwno_a_2nd -> iso3_a_2nd (Non-null: 479 / 2816)
Converted gwno_b -> iso3_b (Non-null: 155 / 2816)


Converted gwno_b_2nd -> iso3_b_2nd (Non-null: 147 / 2816)


Converted gwno_loc -> iso3_loc (Non-null: 2816 / 2816)


## 4. Verification and Sanity Checks

Verify that:
1. **Side B non-state actors** are properly `NaN` in `iso3_b` without generating garbage lists.
2. **Historical states** (Hyderabad, North Yemen, Yugoslavia, USSR) correctly resolved to standard codes.
3. **Multi-state coalitions** (e.g. UK + USA) properly joined with commas.

In [10]:
# 1. Check intrastate civil wars (type_of_conflict == 3)
intrastate_sample = df_clean[df_clean['type_of_conflict'] == 3][
    ['conflict_id', 'year', 'side_a', 'iso3_a', 'side_b', 'gwno_b', 'iso3_b']
].head(5)
print("Sample Intrastate Conflicts (Side B should be NaN):")
display(intrastate_sample)

# 2. Check interstate conflicts (type_of_conflict == 2)
interstate_sample = df_clean[df_clean['type_of_conflict'] == 2][
    ['conflict_id', 'year', 'side_a', 'iso3_a', 'side_b', 'iso3_b']
].head(5)
print("Sample Interstate Conflicts (Side B should be valid ISO3):")
display(interstate_sample)

# 3. Verify historical cases that previously failed
historical_cases = df_clean[df_clean['gwno_a'].isin([751, 678, 316, 345, 816])][
    ['conflict_id', 'year', 'location', 'gwno_a', 'iso3_a']
].drop_duplicates()
print("Historical GW Codes Verification:")
display(historical_cases)

Sample Intrastate Conflicts (Side B should be NaN):


,conflict_id,year,side_a,iso3_a,side_b,gwno_b,iso3_b
0,200,1946,Government of Bolivia,BOL,Popular Revolutionary Movement,NaN,NaN
1,200,1949,Government of Bolivia,BOL,MNR,NaN,NaN
2,200,1952,Government of Bolivia,BOL,MNR,NaN,NaN
3,200,1967,Government of Bolivia,BOL,ELN,NaN,NaN
12,202,1946,Government of China,CHN,PLA,NaN,NaN


Sample Interstate Conflicts (Side B should be valid ISO3):


,conflict_id,year,side_a,iso3_a,side_b,iso3_b
129,214,1946,Government of France,FRA,Government of Thailand,THA
130,215,1946,Government of Albania,ALB,Government of United Kingdom,GBR
141,218,1948,Government of India,IND,Government of Pakistan,PAK
142,218,1964,Government of India,IND,Government of Pakistan,PAK
143,218,1965,Government of India,IND,Government of Pakistan,PAK


Historical GW Codes Verification:


,conflict_id,year,location,gwno_a,iso3_a
353,226,1948,"Hyderabad, India",751,IND
407,230,1948,Yemen (North Yemen),678,YEM
408,230,1962,Yemen (North Yemen),678,YEM
409,230,1963,Yemen (North Yemen),678,YEM
410,230,1964,Yemen (North Yemen),678,YEM
411,230,1965,Yemen (North Yemen),678,YEM
412,230,1966,Yemen (North Yemen),678,YEM
413,230,1967,Yemen (North Yemen),678,YEM
414,230,1968,Yemen (North Yemen),678,YEM
415,230,1969,Yemen (North Yemen),678,YEM


## 5. Export Clean Dataset

Save the cleaned dataset to `data/conflict_clean.csv` for downstream merging with UN Speeches and Dyads.

In [11]:
output_path = file_path.parent / "conflict_clean.csv"
df_clean.to_csv(output_path, index=False)
print(f"Successfully saved clean dataset to {output_path.resolve()}!")

Successfully saved clean dataset to C:\Users\stynz\Desktop\Nová složka (2)\fundamentals_of_data_science\data\conflict_clean.csv!


In [12]:
df_clean

,conflict_id,location,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,territory_name,year,intensity_level,cumulative_intensity,type_of_conflict,start_date,start_prec,start_date2,start_prec2,ep_end,ep_end_date,ep_end_prec,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,region,version,iso3_a,iso3_a_2nd,iso3_b,iso3_b_2nd,iso3_loc
0,200,Bolivia,Government of Bolivia,23,NaN,Popular Revolutionary Movement,719,NaN,2,NaN,1946,2,1,3,1946-07-18,1,1946-07-21,2,1,1946-07-21,NaN,145,NaN,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
1,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,1949,1,1,3,1946-07-18,1,1949-09-15,3,1,1949-09-15,NaN,145,NaN,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
2,200,Bolivia,Government of Bolivia,23,NaN,MNR,720,NaN,2,NaN,1952,1,1,3,1946-07-18,1,1952-04-09,1,1,1952-04-12,NaN,145,NaN,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
3,200,Bolivia,Government of Bolivia,23,NaN,ELN,721,NaN,2,NaN,1967,1,1,3,1946-07-18,1,1967-03-31,3,1,1967-10-16,NaN,145,NaN,NaN,NaN,145,5,26.1,BOL,NaN,NaN,NaN,BOL
4,201,Cambodia (Kampuchea),Government of France,33,NaN,Khmer Issarak,160,NaN,1,Cambodia,1946,1,0,1,1946-08-31,3,1946-08-31,3,0,NaT,NaN,220,NaN,NaN,NaN,811,3,26.1,FRA,NaN,NaN,NaN,KHM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2811,16099,"United Kingdom, United States of America, Yeme...","Government of United Kingdom, Government of Un...","28, 3","Government of Australia, Government of Bahrain...",Government of Yemen (North Yemen),123,NaN,1,"Red Sea, Bab al Mandab and Gulf of Aden",2024,1,0,2,2023-12-31,1,2024-02-03,1,0,NaT,NaN,"200, 2","20, 210, 385, 692, 780, 830, 900, 920",678,NaN,"2, 200, 678","1, 2, 5",26.1,"GBR, USA","CAN, NLD, NOR, BHR, LKA, SGP, AUS, NZL",YEM,NaN,"USA, GBR, YEM"
2812,16099,"United Kingdom, United States of America, Yeme...","Government of United Kingdom, Government of Un...","28, 3","Government of Australia, Government of Bahrain...",Government of Yemen (North Yemen),123,NaN,1,"Red Sea, Bab al Mandab and Gulf of Aden",2025,1,0,2,2023-12-31,1,2024-02-03,1,0,NaT,NaN,"200, 2","20, 210, 385, 692, 780, 830, 900, 920",678,NaN,"2, 200, 678","1, 2, 5",26.1,"GBR, USA","CAN, NLD, NOR, BHR, LKA, SGP, AUS, NZL",YEM,NaN,"USA, GBR, YEM"
2813,16292,Tajikistan,Government of Tajikistan,131,NaN,IS,234,NaN,1,Islamic State,2018,1,0,3,2018-07-30,1,2018-11-08,2,0,NaT,NaN,702,NaN,NaN,NaN,702,3,26.1,TJK,NaN,NaN,NaN,TJK
2814,16292,Tajikistan,Government of Tajikistan,131,NaN,IS,234,NaN,1,Islamic State,2019,1,0,3,2018-07-30,1,2018-11-08,2,1,2019-11-06,NaN,702,NaN,NaN,NaN,702,3,26.1,TJK,NaN,NaN,NaN,TJK
